chargement

In [14]:
import os #dossiers
import json
import random #operations
import hashlib #
import numpy as np

from collections import Counter
from tqdm import tqdm #barre de progression
from PIL import Image
from pycocotools.coco import COCO

# Vérifier le dossier du projet
if os.path.basename(os.getcwd()) == "notebook":
    os.chdir("..")

print("Project directory:", os.getcwd())

random.seed(42)
np.random.seed(42)

# Charger les annotations COCO
coco = COCO("data/annotations/captions_train2017.json")

print("Imports et annotations chargés avec succès.")

Project directory: c:\Users\haman\Downloads\Telegram Desktop\image_pro
loading annotations into memory...
Done (t=0.69s)
creating index...
index created!
Imports et annotations chargés avec succès.


In [15]:
with open("splits/splits_all.json", "r", encoding="utf-8") as f:
    splits = json.load(f)

kb_ids = splits["kb"]
val_ids = splits["val"]
test_ids = splits["test"]
#load the test strata
test_strata = splits["test_strata"]

print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Knowledge Base: 10000
Validation: 100
Test: 100


In [16]:
import shutil

# Sauvegarde de l'ancien fichier (une seule fois)
if not os.path.exists("splits/splits_all_old.json"):
    shutil.copy("splits/splits_all.json", "splits/splits_all_old.json")

# Nouvelle validation : 100 images de train2017 qui ne sont PAS dans la KB
kb_set = set(kb_ids)
candidates = [i for i in sorted(coco.getImgIds()) if i not in kb_set]
val_ids = random.Random(42).sample(candidates, 100)

# On ne change que "val" : kb, test et test_strata restent identiques
splits["val"] = val_ids
with open("splits/splits_all.json", "w", encoding="utf-8") as f:
    json.dump(splits, f)

print("KB :", len(kb_ids), "| Val :", len(val_ids), "| Test :", len(test_ids))
print("Val ∩ KB :", len(set(val_ids) & kb_set))   # doit afficher 0

KB : 10000 | Val : 100 | Test : 100
Val ∩ KB : 0


In [17]:
# Vérifier les tailles
assert len(kb_ids) == 10000
assert len(val_ids) == 100
assert len(test_ids) == 100

assert set(val_ids).isdisjoint(set(kb_ids))
#assert arret le programme n cas une erreur 
# Le test doit être séparé de la KB
assert set(kb_ids).isdisjoint(set(test_ids))

# Le test doit être séparé de la validation
assert set(val_ids).isdisjoint(set(test_ids))

# Vérifier les doublons à l'intérieur de chaque ensemble
assert len(kb_ids) == len(set(kb_ids)) 
assert len(val_ids) == len(set(val_ids))
assert len(test_ids) == len(set(test_ids))

print("Split verification: OK")
print("Validation is separated from KB.")
print("Test is independent from KB and validation.")

Split verification: OK
Validation is separated from KB.
Test is independent from KB and validation.


images corrompues, captions < 5, doublons d’IDs

In [18]:
bad_images = []
wrong_caps = []
empty_caps = []

hashes = {}
duplicate_images = []

modes = Counter()
caps_distribution = Counter()

for iid in tqdm(kb_ids):

    info = coco.loadImgs(iid)[0]
    path = f"data/images/train/{info['file_name']}"

    # Vérifier l'image
    try:
        with Image.open(path) as im:
            im.verify()

        with Image.open(path) as im:
            modes[im.mode] += 1

    except Exception:
        bad_images.append(iid)
        continue

    # Récupérer les captions
    anns = coco.loadAnns(
        coco.getAnnIds(imgIds=[iid])
    )

    n_caps = len(anns)
    caps_distribution[n_caps] += 1

    if n_caps != 5:
        wrong_caps.append((iid, n_caps))

    if any(not a["caption"].strip() for a in anns):
        empty_caps.append(iid)

    # Détecter les doublons exacts
    with open(path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in hashes:
        duplicate_images.append((iid, hashes[file_hash]))
    else:
        hashes[file_hash] = iid


print("\nKNOWLEDGE BASE QUALITY CHECK ")

print("Corrupted or missing images:", len(bad_images))
print("Images with a caption count different from 5:", len(wrong_caps))
print("Images with empty captions:", len(empty_caps))
print("Exact duplicate images:", len(duplicate_images))

print("Verified images:", len(hashes))
print("Image modes:", dict(modes))
print("Captions distribution:", dict(caps_distribution))

100%|██████████| 10000/10000 [00:15<00:00, 657.87it/s]


KNOWLEDGE BASE QUALITY CHECK 
Corrupted or missing images: 0
Images with a caption count different from 5: 23
Images with empty captions: 0
Exact duplicate images: 0
Verified images: 10000
Image modes: {'RGB': 9984, 'L': 16}
Captions distribution: {5: 9977, 6: 22, 7: 1}


In [19]:
coco_val_caps = COCO(
    "data/annotations/captions_val2017.json"
)

coco_val_inst = COCO(
    "data/annotations/instances_val2017.json"
)

print("Test annotations loaded successfully.")

loading annotations into memory...
Done (t=0.10s)
creating index...
index created!
loading annotations into memory...
Done (t=0.36s)
creating index...
index created!
Test annotations loaded successfully.


In [25]:
test_bad = []
test_caps_count = Counter()
test_modes = Counter()

for iid in tqdm(test_ids):

    info = coco_val_caps.loadImgs(iid)[0]
    path = f"data/images/test/{info['file_name']}"

    # Vérifier l'intégrité de l'image
    try:
        with Image.open(path) as im:
            im.verify()

        with Image.open(path) as im:
            test_modes[im.mode] += 1

    except Exception:
        test_bad.append(iid)

    # Vérifier les captions
    anns = coco_val_caps.loadAnns(
        coco_val_caps.getAnnIds(imgIds=[iid])
    )

    test_caps_count[len(anns)] += 1


print("\nTEST QUALITY CHECK ")

print("Corrupted or missing test images:", len(test_bad))
print("Captions distribution:", dict(test_caps_count))
print("Image modes:", dict(test_modes))

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 405.11it/s]


TEST QUALITY CHECK 
Corrupted or missing test images: 0
Captions distribution: {5: 100}
Image modes: {'RGB': 99, 'L': 1}


In [21]:
test_caps = {}

for iid in test_ids:

    anns = coco_val_caps.loadAnns(
        coco_val_caps.getAnnIds(imgIds=[iid])
    )

    test_caps[str(iid)] = [
        a["caption"].strip() for a in anns
    ]


# Créer le dossier
os.makedirs("data/processed/test", exist_ok=True)

# Sauvegarder les captions
with open(
    "data/processed/test/test_captions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        test_caps,
        f,
        indent=4,
        ensure_ascii=False
    )


print("Test captions saved:", len(test_caps))
print("File: data/processed/test/test_captions.json")

Test captions saved: 100
File: data/processed/test/test_captions.json


In [24]:
print("\nFINAL DATASET SUMMARY ")

print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\n SPLIT CHECK ")

print("Validation separated from KB:", 
      set(val_ids).isdisjoint(set(kb_ids)))

print("Test separated from KB:",
      set(kb_ids).isdisjoint(set(test_ids)))

print("\nIMAGE QUALITY ")

print("KB corrupted images:", len(bad_images))
print("Test corrupted images:", len(test_bad))

print("KB images verified:", len(hashes))
print("Test captions saved:", len(test_caps))

print("\nCAPTIONS")

print("KB caption distribution:", dict(caps_distribution))
print("Test caption distribution:", dict(test_caps_count))


FINAL DATASET SUMMARY 
Knowledge Base: 10000
Validation: 100
Test: 100

 SPLIT CHECK 
Validation separated from KB: True
Test separated from KB: True

IMAGE QUALITY 
KB corrupted images: 0


NameError: name 'test_bad' is not defined

In [23]:
print("Knowledge Base:", len(kb_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\nOverlaps:")
print("Validation ∩ KB:", len(set(val_ids) & set(kb_ids)))
print("Test ∩ KB:", len(set(test_ids) & set(kb_ids)))
print("Validation ∩ Test:", len(set(val_ids) & set(test_ids)))

Knowledge Base: 10000
Validation: 100
Test: 100

Overlaps:
Validation ∩ KB: 0
Test ∩ KB: 0
Validation ∩ Test: 0


In [26]:
print("JSON keys:", splits.keys())

print("\nKB examples:", splits["kb"][:5])
print("Validation examples:", splits["val"][:5])
print("Test examples:", splits["test"][:5])

print("\nTest strata:")
print(splits["test_strata"])

JSON keys: dict_keys(['kb', 'val', 'test', 'test_strata', 'note'])

KB examples: [101014, 203463, 462398, 550444, 36447]
Validation examples: [450357, 78354, 17817, 522464, 194027]
Test examples: [129113, 24144, 312406, 273712, 249643]

Test strata:
{'simple (1-3 objets)': [129113, 24144, 312406, 273712, 249643, 161032, 118594, 104198, 484760, 33707, 32081, 109992, 243495, 262440, 27932, 221502, 482477, 245102, 517523, 318080], 'complexe (>= 10 objets)': [5037, 143572, 377588, 303818, 246968, 141671, 185473, 301718, 93154, 84752, 340894, 88462, 324158, 308430, 540502, 233727, 37777, 405691, 474452, 117492], 'multi-categories (>= 5)': [278463, 57597, 395701, 206579, 457078, 449661, 261318, 415194, 146457, 508917, 52996, 30828, 475572, 166918, 558114, 203294, 59598, 171190, 78170, 280710], 'interaction (person + autre)': [253835, 408774, 574702, 331317, 148508, 336053, 323496, 186042, 243034, 60899, 551439, 574297, 154425, 476215, 220732, 148739, 416343, 344029, 245448, 578871], 'context

In [27]:
strata_ids = []

for category, ids in test_strata.items():
    print(category, len(ids))
    strata_ids.extend(ids)

print("Total strata IDs:", len(strata_ids))
print("Unique strata IDs:", len(set(strata_ids)))
print("Same IDs as test:", set(strata_ids) == set(test_ids))

simple (1-3 objets) 20
complexe (>= 10 objets) 20
multi-categories (>= 5) 20
interaction (person + autre) 20
contexte varie (reste) 20
Total strata IDs: 100
Unique strata IDs: 100
Same IDs as test: True


telecharger les 100 de val 

In [29]:
import requests

os.makedirs("data/images/train", exist_ok=True)
download_errors = []

for iid in tqdm(val_ids):
    info = coco.loadImgs(iid)[0]
    path = f"data/images/train/{info['file_name']}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        continue
    try:
        r = requests.get(info["coco_url"], timeout=30)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    except Exception as e:
        download_errors.append((iid, str(e)))

print("Erreurs de téléchargement :", len(download_errors))

100%|██████████| 100/100 [02:52<00:00,  1.72s/it]

Erreurs de téléchargement : 0


In [30]:
val_bad = []
for iid in val_ids:
    p = f"data/images/train/{int(iid):012d}.jpg"
    try:
        with Image.open(p) as im: im.verify()
    except Exception:
        val_bad.append(iid)
print("Manquantes ou corrompues :", len(val_bad))

Manquantes ou corrompues : 0


In [31]:
from PIL import ImageOps
os.makedirs("data/processed/val_images", exist_ok=True)

for iid in val_ids:
    name = f"{int(iid):012d}.jpg"
    with Image.open(f"data/images/train/{name}") as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.save(f"data/processed/val_images/{name}", format="JPEG", quality=95)

print(len(os.listdir("data/processed/val_images")))   # doit afficher 100

100


In [32]:
import pandas as pd
rows = []
for iid in val_ids:
    for a in coco.loadAnns(coco.getAnnIds(imgIds=[iid])):
        rows.append({"image_id": iid, "caption_raw": a["caption"]})

os.makedirs("data/processed/captions", exist_ok=True)
pd.DataFrame(rows).to_csv("data/processed/captions/val_captions.csv", index=False)
print(len(rows))   # environ 500

502
